# Milestone 1: dataset preparation, text embeddings and first samples (Google Colab)

Runtime → Change runtime type → **T4 GPU** (or better). Data and outputs are kept on Google Drive in `MyDrive/IE7615_Project2`, so a disconnected session can resume.

| Step | Script | Output |
|---|---|---|
| 1 | `01_prepare_data.py` | COCO val2017 subset, 512×512 images, `results/dataset_stats.json` |
| 2 | `02_encode_captions.py` | CLIP ViT-L/14 embeddings for every caption |
| 3 | `03_embedding_checks.py` | caption similarity + text→image retrieval, `results/embedding_checks.json` |
| 4 | `04_sample_prompts.py` | 5 prompt→image samples, external-embedding conditioning check |

**Pipeline**

```
Text prompt ──► CLIP tokenizer ──► CLIP text encoder ──► embeddings (77×768) ──► SD v1.5 U-Net + DDIM ──► VAE decoder ──► image (512×512)
                (BPE, 77 tokens)   (ViT-L/14, 12 layers)   cross-attention context   (30 steps, CFG 7.5)     (64×64×4 latent → pixels)
```

Steps 1–3 prepare and check the text side; step 4 runs the whole chain from prompt to image.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IE7615_Project2'
except Exception as e:
    print('Google Drive not mounted, outputs stay in /content/outputs:', e)
    DRIVE = '/content/outputs'
REPO = 'https://github.com/anshuman-dataverse/text-to-image-generation.git'
NUM_IMAGES = 5000
os.makedirs(f'{DRIVE}/data', exist_ok=True)

In [ ]:
%cd /content
!test -d project || git clone -q $REPO project
# fetch + hard reset: Colab-generated results/ must not block updating to the latest code
!cd project && git remote set-url origin $REPO && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
!pip -q install -r project/requirements.txt
# Colab's preinstalled torchao is too old for the latest diffusers (FqnToConfig import error); we don't use it
!pip -q uninstall -y torchao
# keep the COCO download and processed data on Drive
!rm -rf project/milestone1/data && ln -sfn "$DRIVE/data" project/milestone1/data

## 1. Download COCO val2017, clean captions, subset and resize

`01_prepare_data.py` downloads COCO val2017 (5,000 images, 25,014 raw captions) and:
- cleans captions (whitespace, capitalization, one trailing period), drops captions under 3 words and exact duplicates per image (14 removed, leaving 25,000);
- resizes each image on its shorter side to 512 and center-crops to 512×512 (SD v1.5's native resolution);
- builds the **image–caption pair subset**: one caption per image, picked at random with seed 42 → **5,000 pairs**, saved as `results/pairs_manifest.csv` (image_id, caption_id, file_name, caption);
- checks the subset for missing or unreadable images, wrong sizes, empty captions and duplicate image or caption IDs (`results/subset_checks.json`), and stops if any check fails.

In [ ]:
%cd /content/project/milestone1/scripts

import os, subprocess
from pathlib import Path
from IPython.display import Image, display

def run(script, *args):
    """Run a milestone script, stream its output here and to ../runs/<script>.log, stop on failure."""
    log = Path('../runs') / (Path(script).stem + '.log')
    log.parent.mkdir(exist_ok=True)
    commit = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
    print('code version:', commit)
    with open(log, 'w') as f:
        f.write(f'code version: {commit}\n')
        p = subprocess.Popen(['python', '-u', script, *map(str, args)], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            f.write(line)
    if p.wait():
        raise RuntimeError(f'{script} failed (exit code {p.returncode}). The error is printed above and saved in {log}')

def show(*names):
    for n in names:
        path = Path('../screenshots') / n
        display(Image(str(path))) if path.exists() else print(f'missing: {path} (the script before this cell did not finish)')

In [ ]:
run('01_prepare_data.py', '--num-images', NUM_IMAGES)

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open('../results/subset_checks.json')), indent=2))
pd.read_csv('../results/pairs_manifest.csv').head(10)

In [ ]:
show('dataset_examples.png', 'caption_lengths.png')

## 2. Text → embedding pipeline (CLIP ViT-L/14)

`02_encode_captions.py` runs every caption through the CLIP tokenizer (BPE, padded/truncated to 77 tokens) and the CLIP ViT-L/14 text encoder, the same encoder SD v1.5 uses internally, and caches the pooled embeddings. The cell after it shows the two outputs for one prompt:
- **per-token embeddings, 77×768**: the context the U-Net cross-attends to at every denoising step;
- **pooled embedding, 768**: the EOS token, used for the similarity and retrieval checks in step 3.

In [ ]:
run('02_encode_captions.py')

In [ ]:
import sys, torch
sys.path.insert(0, '.')
from t2i.embeddings import TextEncoder
enc = TextEncoder('openai/clip-vit-large-patch14')
caption = 'A man riding a bicycle down a city street.'
tok = enc.tokenize([caption])
print(enc.tokenizer.convert_ids_to_tokens(tok['input_ids'][0][:int(tok['attention_mask'].sum())]))
out = enc.encode([caption])
print('per-token (U-Net cross-attention context):', tuple(out['hidden'].shape))
print('pooled (EOS token):', tuple(out['pooled'].shape))
del enc; torch.cuda.empty_cache()

## 3. Embedding sanity checks

`03_embedding_checks.py` checks that the embeddings carry the caption's meaning before we use them as conditioning:
1. **Caption similarity**: captions of the same image should be closer (cosine) than captions of different images. Expected: about 0.66 vs. 0.23, pairwise AUC about 0.985.
2. **Text→image retrieval**: CLIP's text and image towers share one space, so each caption should find its own image among 1,000. Expected Recall@1 / @5 / @10: about 54.9% / 81.1% / 89.7%.

In [ ]:
run('03_embedding_checks.py')

In [ ]:
show('caption_similarity.png', 'retrieval_examples.png')

## 4. Five prompt → image test runs (Stable Diffusion v1.5, DDIM, 30 steps, guidance 7.5)

`04_sample_prompts.py` runs the full chain: prompt → CLIP tokenizer → text encoder → 77×768 embeddings → U-Net denoising with the DDIM scheduler and classifier-free guidance (30 steps, w = 7.5) → VAE decoder → 512×512 image. Prompt *i* uses seed *i* (0–4), so the images are reproducible.

It then encodes the same prompts with our own `TextEncoder` and passes the tensors to the U-Net as `prompt_embeds`. Identical images confirm that our embeddings condition the model exactly like SD's built-in text path.

In [ ]:
run('04_sample_prompts.py')

In [ ]:
# Each sample with its prompt and seed
meta = json.load(open('../results/samples/metadata.json'))
print(f"{meta['model_id']} | {meta['scheduler']} | {meta['steps']} steps | guidance {meta['guidance_scale']} | {meta['gpu']} | {meta['seconds_per_image']} s/image")
for s in meta['samples']:
    print(f"\nseed {s['seed']}: {s['prompt']}")
    display(Image(f"../results/samples/{s['file']}", width=320))

In [ ]:
show('milestone1_samples.png', 'conditioning_check.png')

**Observations.** Single-subject scenes (cyclist, pasta) come out photorealistic and match the prompt. Quality drops as prompts list more objects and attributes: the frisbee is missing (object omission), the "blue car" turns into a blue-fronted second bus (attribute leakage), and "at sunset" is ignored. These are known limits of CLIP-conditioned diffusion on compositional prompts, and Milestone 2 tests whether guidance scale and scheduler choice help. Faces, hands and text in the images are also unreliable at 512×512.

## 5. Save outputs to Drive
Download `milestone1_outputs.zip`, unzip into `milestone1/` of the local repo and commit.

In [ ]:
%cd /content/project/milestone1
!zip -qr "$DRIVE/milestone1_outputs.zip" results screenshots runs
!ls -la "$DRIVE"